In [1]:
# Import libraries
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import openpyxl

# Import Class
from pathlib import Path

In [2]:
# Set Path to current working directoryu
cwd = Path.cwd()

# Set filesystem paths
ROOT = cwd.parent if cwd.name == "notebooks" else cwd

In [3]:
# Establish paths
ROOT

DATA_RAW = ROOT / "data" / "raw"
DATA_RAW

DATA_PROCESSED = ROOT / "data" / "processed"

FIGURES = ROOT / "figures"

In [4]:
# Set scope in years of available data
YEAR_MIN = 2015
YEAR_MAX = 2023

In [5]:
# # Verify all paths
# print("ROOT:", ROOT)
# print("RAW DATA:", DATA_RAW)
# print("PROCESSED DATA:", DATA_PROCESSED)
# print("FIGURES:", FIGURES)

In [6]:
# Load raw data
RAW_DEMOGRAPHIC_ASPECTS_FILE = DATA_RAW / "Demographic-aspects-2023.xlsx"

In [7]:
# Create "if statement" to test for errors
if not RAW_DEMOGRAPHIC_ASPECTS_FILE.exists():
    raise FileNotFoundError

In [8]:
# Set variable for wide format
df_wide = pd.read_excel(RAW_DEMOGRAPHIC_ASPECTS_FILE, header=1)

In [9]:
# Filter the data into our two groups
males = df_wide[df_wide["Key Demographic aspects"] == "Males"]
females = df_wide[df_wide["Key Demographic aspects"] == "Females"]

In [15]:
# Calculate the yearly change (slicing 2015 to 2023)
# Using.diff(axis=1) to subtract each year from the previous one
yearly_change_males = males.loc[:, 2015:2023].diff(axis=1)
yearly_change_females = females#.loc[:, 2015:2023].diff(axis=1)

In [16]:
yearly_change_males

,2015,2016,2017,2018,2019,2020,2021,2022,2023
0,NaN,153.0,-63.0,113.0,3.0,-585.0,-266.0,-224.0,119.0


In [11]:
# # Combine them into one comparison table
comparison_df = pd.concat([yearly_change_males, yearly_change_females], keys=["Males", "Females"])

In [12]:
# Clean up: drop the extra index level and fill the 2015 NaNs with 0
comparison_df = comparison_df.droplevel(1).fillna(0)

In [13]:
# Convert values and assign new variables
comparison_df.columns = comparison_df.columns.astype(int)

years = comparison_df.columns
yearly_change_males = comparison_df.loc["Males"].values
yearly_change_females = comparison_df.loc["Females"].values

yearly_change_total = yearly_change_males + yearly_change_females

ValueError: invalid literal for int() with base 10: 'Key Demographic aspects'

In [ ]:
# Create plot
plt.figure(figsize=(12, 7))

plt.plot(years, yearly_change_males, marker="o", label="Males")
#plt.plot(years, yearly_change_females, marker="o", label="Females")
plt.plot(years, yearly_change_total, marker="o", linewidth=2.5, label="Total Population")

plt.yticks(np.arange(-1400, 501, 100))

plt.axhline(0, color="black", linestyle="--", linewidth=1)

# Shaded area for extra clarity
plt.axvspan(2019.5, 2021.5, alpha=0.15)

# Added label
plt.text(2020.5, 250, "COVID-19 disruption period",
         ha="center", fontsize=10)

# Arrow (for added graphic design)
plt.annotate(
    "Sharp decline in 2020",
    xy=(2020, -1275),
    xytext=(2018.8, 200),
    arrowprops=dict(arrowstyle="->", alpha=0.75)
)

plt.title("Annual Population Change in Aruba by Sex, 2015–2023")
plt.xlabel("Year")
plt.ylabel("Annual Growth / Decline (Absolute)")
plt.grid(True)

plt.legend()

# Save figure
plt.savefig("Annual_Pop_Change-Sex")
plt.show()